# Stripping Metadata from Strings

This demo notebook demonstrates how to strip line-initial segment identifiers from a plaintext corpus. We will use [ECHOE](https://github.com/ECHOEProject/echoe) as an example.

You'll want to run this notebook from within `../project/` where it is untracked.

## The Problem

Text corpora often incorporate segment identifiers (e.g. chapter and verse references), which in many NLP applications represent noise. This notebook showcases two ways of getting rid of them. First, we'll load lines of text into a dictionary in which we separate out the references as dictionary keys. Afterwards, we'll write the lines to disk again, this time suppressing the identifiers altogether.

For this template we'll rely on the `pathlib` library for disk operations. This is one option among a number of eligible candidates, including `os` and `glob`, so take your pick:

In [1]:
from pathlib import Path
from git import Repo

The following cell is just to ascertain that ECHOE is present and up to date in `../corpora/echoe/`:

In [2]:
# HTTPS clone point:
remote = 'https://github.com/ECHOEProject/echoe.git'
# Desired target folder name:
local = Path.cwd().parent / 'corpora' / 'echoe'
# Only clone if the target folder doesn't already exist:
if not(local.exists()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

Now just to define our source and target folders:

In [3]:
plaintext_path = local / 'plaintext'
target_path = Path.cwd().parent / 'corpora' / 'echoe-bare'
Path(target_path).mkdir(parents=True, exist_ok=True)
echoe_plain = Path(plaintext_path)

For our first operation, we'll assume you want to work with the plaintext corpus and store segment identifiers separately from the document text. In the below implementation, we are assuming that segment identifiers are printed at the start of each line and followed by a spaced colon (`: `), as in ECHOE's plaintext corpus. However, the below code also chooses to discard any lines that lack such a sequence. In ECHOE, that means not only that empty lines are discarded, but rubrics (i.e. headings) as well. If you need to retain either or both, you'll want to modify the code or come up with a different approach:

In [4]:
# We will store the corpus in a Python dictionary. For this first example,
# we will use the filename (minus extension) as the dictionary key:
echoe = dict()
for file in echoe_plain.glob('*txt'):
    # We will then create a further dictionary for each file, with segment
    # identifiers as keys:
    new_doc = dict()
    filename = file.name.rstrip('.txt')
    # We'll process it one line at a time
    # (ECHOE prints one sentence-like segment to a line, each with its own identifier):
    lines = open(file).read().splitlines()
    for line in lines:
        if ': ' in line:
            # In ECHOE's plaintext corpus, colons only ever occur between identifiers and text.
            # However, just to be sure, we will split each line a maximum of once:
            pair = line.split(': ', 1)
            # We then add the segment to the document dictionary, callable by its identifier:
            new_doc[pair[0]] = pair[1].rstrip()
    echoe[filename] = new_doc

Now if we want to print a document in full, we can print it either with or without segment identifiers, depending on our needs:

In [5]:
for k,v in echoe['402.bii'].items():
    print(f"{k}: {v}")

402.bii.1: nemo christianorum paganas superstitiones intendat sed gentilium inquinamenta omnia omnimodo contemnat
402.bii.2: eala mycel is nydþearf manna gehwylcum þæt he wið deofles larswice warnige symle
402.bii.3: and þæt he hæðenscype georne æfre forbuge þæs þe he gedon mæge
402.bii.4: and gyf hit geweorðe þæt cristenman æfre heonanforð ahwar heðendom begange oððon ahwar on lande idola weorðige gebete þæt deope for gode and for worolde
402.bii.7: and gyf wiccean oððe wigelearas horingas oððe horcwenan morðwyrhtan oððe mansworan innan þysan earde weorðan agytene fyse hy man georne ut of þysan earde and clænsige þas þeode oððon on earde forfare hy mid ealle
402.bii.8: butan he geswicon and þe deoppor gebetan
402.bii.9: and do man swa hit þearf is
402.bii.10: manfulra dæda on æghwylcan ende styre man swyðe
402.bii.11: her syndan on earde godcundnessæ wiðersacan and godes lage oferhogan
402.bii.12: manslagan and mægslagan
402.bii.13: cyrichatan and sacerdbanan
402.bii.14: hadbrecan and

In [6]:
for line in echoe['402.biii'].values():
    print(line)

a christo enim cristiani sunt nominati christus autem capud nostrum est
et nos menbra eius
crist is ealra cristenra manna heafod
and ealle cristene men syndon to cristes limum getealde gyf hy heora drihtne gecwemað mid rihte
and hy scylan swyþe georne cristendom æfre healdan mid rihte and cristes cyrcan secan gelome heom sylfum to þearfe
and cristes gerihta rihtlice gelæstan
and þæt is an ærest þæt man geteoðige æghwylce geare þæt þæt god sende þonne on geare folce to þearfe on corne and on flexe on gewelhwylcon wæstme
and arise seo æcerteoðung a be ðam þe seo sulh þone teoðan æcer ær geeode
be godes miltse and be ðæs cynges and be ealles cristenes folces
and be ðære steore þe eadgar cyng gelagode
and sy ælcere geogoðe teoðung gelæst be pentecosten be wite
and eorðwæstma be ealra halgena mæssan
and romfeoh gelæste man æghwylce geare be petres mæssan
and se ðe hit ne gelæste sylle þærtoeacan xxx peningas ringe to rome
and gylde þam cynge on engla lage cxx scillingas
and cyricsceat gelæs

And if we want to print a single segment, we can do so as follows:

In [7]:
echoe['049B.01']['49B.1.1']

'adam se æresta man wæs gescapen on neorxnawonge'

That's a pretty awkward expression though, and in the case of ECHOE, segment identifiers are unique, so the nesting dictionaries are unnecessary. Therefore let's repeat the exact same process but now use only a single dictionary for the entire corpus:

In [8]:
echoe = dict()
for file in echoe_plain.glob('*txt'):
    new_doc = dict()
    filename = file.name.rstrip('.txt')
    lines = open(file).read().splitlines()
    for line in lines:
        if ': ' in line:
            pair = line.split(': ', 1)
            echoe[pair[0]] = pair[1].rstrip()

Now we can access the same content as follows:

In [9]:
echoe['49B.1.1']

'adam se æresta man wæs gescapen on neorxnawonge'

Finally, let's print everything back to disk, this time including rubrics and empty lines, but without any identifiers, so we can use the corpus in NLP analysis without further preprocessing:

In [10]:
for file in echoe_plain.glob('*txt'):
    outlines = []
    outfile = target_path / file.name
    inlines = open(file).read().splitlines()
    for line in inlines:
        if ': ' in line:
            line = line.split(': ', 1)[1]
        outlines.append(line)
    outdoc = '\n'.join(outlines)
    with open(outfile, 'w') as f:
        f.write(outdoc)
        